# 🧠 Glioma Segmentation — 2D U-Net + BraTS2020
**Paper:** *Optimizing Magnetic Resonance Image Segmentation Through Scalable Deep Learning and Hierarchical Data Management*  
**Autores:** Lídices Reyes-Hung, Gabriel Trinke, Ismael Soto — USACH, Chile

---
### ✅ Antes de correr
1. **Runtime → Change runtime type → T4 GPU**
2. Ten listo tu `kaggle.json`
3. Ejecuta las celdas **en orden**

### 📁 Todo se guarda en Drive
```
Mi unidad/brats2020_unet/
  best_unet2d.keras         ← mejor modelo
  history.npy               ← historial
  test_files.npy            ← rutas test set
  learning_curves.png       ← Fig. 3 del paper
  qualitative_results.png   ← Fig. 4 del paper
```


## 📦 CELDA 1 — Instalar dependencias

In [ ]:
!pip install -q kaggle h5py scikit-learn scipy matplotlib numpy

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

## 💾 CELDA 2 — Montar Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/brats2020_unet'
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f'Drive montado ✓')
print(f'Carpeta: {DRIVE_DIR}')
print(f'\nContenido actual:')

archivos = sorted(os.listdir(DRIVE_DIR))
if len(archivos) == 0:
    print('  (carpeta vacía — primer uso)')
else:
    for f in archivos:
        fpath = os.path.join(DRIVE_DIR, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / 1e6
            print(f'  {f:<40} {size:>8.1f} MB')
        else:
            print(f'  {f:<40} (carpeta)')

## ⚙️ CELDA 3 — Configuración global

In [ ]:
import os
import glob
import numpy as np
import h5py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import directed_hausdorff

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, Activation,
    MaxPooling2D, Conv2DTranspose, concatenate, Dropout
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
)

# ─── Rutas Drive ──────────────────────────────────────────────────
DRIVE_DIR    = '/content/drive/MyDrive/brats2020_unet'
MODEL_PATH   = os.path.join(DRIVE_DIR, 'best_unet2d.keras')
HISTORY_PATH = os.path.join(DRIVE_DIR, 'history.npy')
TEST_PATH    = os.path.join(DRIVE_DIR, 'test_files.npy')
CURVES_PATH  = os.path.join(DRIVE_DIR, 'learning_curves.png')
QUAL_PATH    = os.path.join(DRIVE_DIR, 'qualitative_results.png')

# ─── Hiperparámetros del paper ────────────────────────────────────
DATA_DIR   = '/content/brats2020_h5'
IMG_SIZE   = 128
N_CHANNELS = 4
BATCH_SIZE = 16   # T4 aguanta 16 (paper usa 16)
EPOCHS     = 15
LR         = 1e-3
SEED       = 42
THRESHOLD  = 0.5

print('Configuración cargada ✓')
print(f'  BATCH_SIZE={BATCH_SIZE} | EPOCHS={EPOCHS} | LR={LR} | IMG_SIZE={IMG_SIZE}')

## 📥 CELDA 4 — Descargar dataset del paper desde Kaggle
> Dataset: `awsaf49/brats2020-training-data` — 57,195 archivos `.h5` pre-sliced  
> ⚠️ Salta esta celda si el dataset ya está en `/content/brats2020_h5`

In [ ]:
import os, glob

h5_check = glob.glob('/content/brats2020_h5/**/*.h5', recursive=True)

if len(h5_check) > 0:
    print(f'Dataset ya existe → {len(h5_check)} archivos .h5 encontrados')
    print('Salta a Celda 5')

else:
    from google.colab import files

    # ── Configurar Kaggle ────────────────────────────────────────
    print('Sube tu kaggle.json:')
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('Kaggle configurado ✓')

    # ── Descargar ────────────────────────────────────────────────
    os.makedirs('/content/brats2020_h5', exist_ok=True)
    print('\nDescargando dataset (~4 GB)...')
    !kaggle datasets download -d awsaf49/brats2020-training-data \
        -p /content/brats2020_h5

    print('\nDescomprimiendo...')
    !unzip -q /content/brats2020_h5/*.zip -d /content/brats2020_h5
    !rm /content/brats2020_h5/*.zip

    h5_final = glob.glob('/content/brats2020_h5/**/*.h5', recursive=True)
    print(f'\n✓ Archivos .h5: {len(h5_final)}')
    print('Contenido:')
    !ls /content/brats2020_h5/

## 🔍 CELDA 5 — Verificar dataset y construir lista de archivos

In [ ]:
all_files = sorted(glob.glob(
    os.path.join(DATA_DIR, '**', '*.h5'), recursive=True
))

if len(all_files) == 0:
    raise RuntimeError('No se encontraron archivos .h5 — vuelve a correr Celda 4')

print(f'Total archivos .h5: {len(all_files)}')
print(f'Ejemplo: {all_files[0]}')

# Verificar estructura interna
with h5py.File(all_files[0], 'r') as f:
    print(f'\nEstructura del archivo H5:')
    for k in f.keys():
        print(f'  {k}: shape={f[k].shape} | dtype={f[k].dtype}')

## 📂 CELDA 6 — Función de lectura H5 + Generador (lazy loading)

In [ ]:
# ─── Leer un slice H5 (igual que gioma.py) ────────────────────────
def read_h5_sample(path, img_size=128):
    with h5py.File(path, 'r') as f:
        x = f['image'][()].astype(np.float32)  # (H, W, 4)
        y = f['mask'][()].astype(np.float32)   # (H, W, 3) o (H, W, 1)

    # Máscara → binaria
    if y.ndim == 3 and y.shape[-1] == 3:
        y = (np.sum(y, axis=-1) > 0).astype(np.float32)[..., np.newaxis]
    elif y.ndim == 2:
        y = (y > 0).astype(np.float32)[..., np.newaxis]
    elif y.ndim == 3 and y.shape[-1] == 1:
        y = (y > 0).astype(np.float32)
    else:
        raise RuntimeError(f'Formato máscara no soportado: {y.shape}')

    x = tf.image.resize(x, [img_size, img_size], method='bilinear').numpy()
    y = tf.image.resize(y, [img_size, img_size], method='nearest').numpy()
    y = (y > 0).astype(np.float32)
    return x, y


# ─── Generador lazy-loading (igual que gioma.py) ──────────────────
class SliceH5Generator(tf.keras.utils.Sequence):
    def __init__(self, file_list, batch_size=16, shuffle=True, img_size=128):
        self.file_list  = np.array(file_list)
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.img_size   = img_size
        self.indices    = np.arange(len(self.file_list))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.file_list) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx   = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]
        batch_files = self.file_list[batch_idx]
        X, Y = [], []
        for path in batch_files:
            x, y = read_h5_sample(path, self.img_size)
            X.append(x)
            Y.append(y)
        return np.array(X, np.float32), np.array(Y, np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


# ─── Verificar lectura ────────────────────────────────────────────
x0, y0 = read_h5_sample(all_files[0], IMG_SIZE)
print(f'Shape imagen : {x0.shape}')
print(f'Shape máscara: {y0.shape}')
print(f'Rango imagen : [{x0.min():.2f}, {x0.max():.2f}]')
print(f'Voxels tumor : {y0.sum():.0f} / {y0.size}')

## ✂️ CELDA 7 — Split train / val / test + Generadores

In [ ]:
# Split igual que gioma.py
train_files, test_files = train_test_split(
    all_files, test_size=0.10, random_state=SEED
)
train_files, val_files = train_test_split(
    train_files, test_size=0.15/0.90, random_state=SEED
)

# Guardar test_files en Drive (para evaluar aunque se caiga la sesión)
np.save(TEST_PATH, np.array(test_files))

print(f'Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}')
print(f'Test files guardados → {TEST_PATH}')

train_gen = SliceH5Generator(train_files, BATCH_SIZE, shuffle=True,  img_size=IMG_SIZE)
val_gen   = SliceH5Generator(val_files,   BATCH_SIZE, shuffle=False, img_size=IMG_SIZE)

print(f'\nPasos por época → train: {len(train_gen)} | val: {len(val_gen)}')

# Verificar un batch
Xb, yb = train_gen[0]
print(f'Batch imagen : {Xb.shape}')
print(f'Batch máscara: {yb.shape}')

## 🏗️ CELDA 8 — Arquitectura 2D U-Net (Fig. 2 del paper)

In [ ]:
# ─── Loss y métricas ──────────────────────────────────────────────
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    yt = tf.keras.backend.flatten(tf.cast(y_true, tf.float32))
    yp = tf.keras.backend.flatten(tf.cast(y_pred, tf.float32))
    intersection = tf.reduce_sum(yt * yp)
    return (2.0*intersection+smooth) / (tf.reduce_sum(yt)+tf.reduce_sum(yp)+smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)

def specificity_np(y_true, y_pred):
    tn = np.sum((y_true==0) & (y_pred==0))
    fp = np.sum((y_true==0) & (y_pred==1))
    return (tn+1e-6) / (tn+fp+1e-6)

def hausdorff_distance_binary(y_true, y_pred):
    pts_t = np.argwhere(y_true > 0)
    pts_p = np.argwhere(y_pred > 0)
    if len(pts_t)==0 and len(pts_p)==0: return 0.0
    if len(pts_t)==0 or  len(pts_p)==0: return np.nan
    return max(directed_hausdorff(pts_t,pts_p)[0],
               directed_hausdorff(pts_p,pts_t)[0])


# ─── Bloques convolucionales ──────────────────────────────────────
def conv_block(x, filters, dropout_rate=0.1):
    x = Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)
    return x


# ─── U-Net 2D ~7.8M parámetros ────────────────────────────────────
def build_unet(input_shape=(128, 128, 4)):
    inputs = Input(shape=input_shape)

    c1 = conv_block(inputs, 32,  0.1);  p1 = MaxPooling2D(2)(c1)
    c2 = conv_block(p1,     64,  0.1);  p2 = MaxPooling2D(2)(c2)
    c3 = conv_block(p2,     128, 0.2);  p3 = MaxPooling2D(2)(c3)
    c4 = conv_block(p3,     256, 0.2);  p4 = MaxPooling2D(2)(c4)
    bn = conv_block(p4,     512, 0.3)

    u6 = Conv2DTranspose(256, 2, strides=2, padding='same')(bn)
    c6 = conv_block(concatenate([u6, c4]), 256, 0.2)
    u7 = Conv2DTranspose(128, 2, strides=2, padding='same')(c6)
    c7 = conv_block(concatenate([u7, c3]), 128, 0.2)
    u8 = Conv2DTranspose(64,  2, strides=2, padding='same')(c7)
    c8 = conv_block(concatenate([u8, c2]), 64,  0.1)
    u9 = Conv2DTranspose(32,  2, strides=2, padding='same')(c8)
    c9 = conv_block(concatenate([u9, c1]), 32,  0.1)

    outputs = Conv2D(1, 1, activation='sigmoid')(c9)
    return Model(inputs, outputs, name='UNet2D_H5Slices')


# ─── Cargar modelo si existe en Drive, si no construir nuevo ──────
if os.path.exists(MODEL_PATH):
    print(f'Modelo existente en Drive → cargando...')
    model = tf.keras.models.load_model(
        MODEL_PATH,
        custom_objects={'dice_loss': dice_loss,
                        'dice_coefficient': dice_coefficient}
    )
    print('Modelo cargado ✓ — puedes saltar directo a Celda 10')
else:
    model = build_unet(input_shape=(IMG_SIZE, IMG_SIZE, N_CHANNELS))
    model.compile(
        optimizer=Adam(learning_rate=LR),
        loss=dice_loss,
        metrics=[dice_coefficient,
                 tf.keras.metrics.Recall(name='sensitivity')]
    )
    print('Modelo nuevo construido ✓')

model.summary()
print(f'\nParámetros entrenables: {model.count_params():,}')

## 🚀 CELDA 9 — Entrenamiento (paper §IV-D)
> Tiempo estimado en T4: ~6-8 horas (15 épocas × ~3,200 pasos)  
> El mejor modelo se guarda en Drive automáticamente

In [ ]:
if os.path.exists(HISTORY_PATH):
    print('Ya existe historial en Drive.')
    print('Si quieres re-entrenar borra history.npy y best_unet2d.keras de Drive.')
    history_dict = np.load(HISTORY_PATH, allow_pickle=True).item()

else:
    callbacks = [
        ModelCheckpoint(
            MODEL_PATH,
            monitor='val_dice_coefficient', mode='max',
            save_best_only=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=3, min_lr=1e-6, verbose=1
        ),
        EarlyStopping(
            monitor='val_loss', patience=7,
            restore_best_weights=True, verbose=1
        ),
    ]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    history_dict = history.history
    np.save(HISTORY_PATH, history_dict)
    print(f'\n✓ Modelo guardado   → {MODEL_PATH}')
    print(f'✓ Historial guardado → {HISTORY_PATH}')

## 📈 CELDA 10 — Curvas de aprendizaje (Fig. 3 del paper)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_dict['loss'],     label='train_loss',  linewidth=2)
axes[0].plot(history_dict['val_loss'], label='val_loss',    linewidth=2)
axes[0].set_title('Dice Loss', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_dict['dice_coefficient'],     label='train_dice', linewidth=2)
axes[1].plot(history_dict['val_dice_coefficient'], label='val_dice',   linewidth=2)
axes[1].set_title('Dice Coefficient', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CURVES_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Figura guardada → {CURVES_PATH}')

## 📊 CELDA 11 — Evaluación cuantitativa test set (Tabla II del paper)
> Objetivo: **DSC≈0.884 | Sensitivity≈0.851 | Specificity≈0.992 | HD≈4.2mm**

In [ ]:
# Cargar test_files desde Drive
test_files_eval = np.load(TEST_PATH, allow_pickle=True).tolist()
print(f'Evaluando {len(test_files_eval)} muestras...')

dices, recalls, specs, hds = [], [], [], []

for path in test_files_eval:
    x, y_true = read_h5_sample(str(path), IMG_SIZE)
    y_prob = model.predict(x[np.newaxis,...], verbose=0)[0]
    y_pred = (y_prob >= THRESHOLD).astype(np.uint8)

    yt = y_true[...,0].astype(np.uint8)
    yp = y_pred[...,0].astype(np.uint8)

    tp   = np.sum((yt==1)&(yp==1))
    fn   = np.sum((yt==1)&(yp==0))
    fp   = np.sum((yt==0)&(yp==1))
    e    = 1e-6

    dices.append((2*tp+e)/(2*tp+fp+fn+e))
    recalls.append((tp+e)/(tp+fn+e))
    specs.append(specificity_np(yt, yp))
    hd = hausdorff_distance_binary(yt, yp)
    if not np.isnan(hd):
        hds.append(hd)

print('\n' + '='*54)
print('  RESULTADOS TEST SET  (Tabla II del paper)')
print('='*54)
print(f'  Dice:        {np.mean(dices):.4f}   (paper: 0.884)')
print(f'  Sensitivity: {np.mean(recalls):.4f}   (paper: 0.851)')
print(f'  Specificity: {np.mean(specs):.4f}   (paper: 0.992)')
print(f'  Hausdorff:   {np.mean(hds):.4f} mm (paper: 4.2 mm)')
print('='*54)

## 🖼️ CELDA 12 — Resultados cualitativos estilo paper (Fig. 4)
Columna **(a)** FLAIR | **(b)** Ground Truth superpuesto | **(c)** Predicción superpuesta

In [ ]:
def normalize_display(img):
    mn, mx = img.min(), img.max()
    return img if mx-mn < 1e-8 else (img-mn)/(mx-mn)

def overlay_mask(base_gray, mask, color=(1.0, 0.2, 0.1), alpha=0.50):
    rgb = np.stack([base_gray]*3, axis=-1)
    rgb[mask.astype(bool)] = (
        (1-alpha)*rgb[mask.astype(bool)] + alpha*np.array(color)
    )
    return np.clip(rgb, 0, 1)


# Seleccionar muestras con tumor visible
N_VIS  = 4
chosen = []
for path in test_files_eval:
    x, y = read_h5_sample(str(path), IMG_SIZE)
    if y.sum() > 100:
        chosen.append((x, y))
    if len(chosen) == N_VIS:
        break

fig, axes = plt.subplots(N_VIS, 3,
                          figsize=(9, 3*N_VIS),
                          facecolor='black')

for col, title in enumerate(['FLAIR', 'Ground\nTruth', 'Axial\n(Prediction)']):
    axes[0, col].set_title(title, color='white', fontsize=11, pad=6)

for row, (x, y) in enumerate(chosen):
    yp     = model.predict(x[np.newaxis,...], verbose=0)[0,...,0]
    yp_bin = (yp >= THRESHOLD).astype(np.float32)
    flair  = normalize_display(x[:,:,0])

    imgs  = [flair, overlay_mask(flair, y[...,0]), overlay_mask(flair, yp_bin)]
    cmaps = ['gray', None, None]

    for col in range(3):
        axes[row, col].imshow(imgs[col], cmap=cmaps[col], vmin=0, vmax=1)
        axes[row, col].axis('off')
        axes[row, col].set_facecolor('black')

    # DSC por slice
    tp     = np.sum((yp_bin==1)&(y[...,0]==1))
    dsc_sl = (2*tp+1e-6)/(2*tp+np.sum(yp_bin!=y[...,0])+1e-6)
    axes[row,2].text(
        0.98, 0.02, f'DSC {dsc_sl:.3f}',
        transform=axes[row,2].transAxes,
        ha='right', va='bottom', fontsize=8, color='white',
        bbox=dict(facecolor='black', alpha=0.5, pad=2, edgecolor='none')
    )

plt.subplots_adjust(wspace=0.03, hspace=0.03,
                    left=0.01, right=0.99,
                    top=0.93,  bottom=0.01)
plt.savefig(QUAL_PATH, dpi=180, facecolor='black', bbox_inches='tight')
plt.show()
print(f'✓ Figura guardada → {QUAL_PATH}')

## ✅ CELDA 13 — Verificar todo lo guardado en Drive

In [ ]:
print(f'Archivos en Drive → {DRIVE_DIR}/')
print()
for fname in sorted(os.listdir(DRIVE_DIR)):
    fpath = os.path.join(DRIVE_DIR, fname)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f'  {fname:<40} {size:>8.1f} MB')

print('\n✓ Experimento completo. Todo guardado en Drive.')